# SOLA on H2 functions: refine the numerics of a fixed problem

The rays, truth, noisy observations and physical disk-average target are identical
to the pixel notebook. The model space is the restriction Sobolev space
$H^2([0,1]^2)$ with length scale $0.1$. Ray integration is a bounded functional
there. No model norm bound or probability distribution is asserted.

The inner product specifies how functionals are compared. This changes the
selection of X; the transformed observation equation remains exact for the
selected weights. The experiment refines quadrature, not the model space.

In [ ]:
%matplotlib inline
import json, time, socket, hashlib, subprocess, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import sola_demo as S
P = S.P
OUT = Path('output'); OUT.mkdir(exist_ok=True)
src, sta, p, q = P.geometry()
d_clean, d = P.data()
truth = S.target_truth()
print('machine:', socket.gethostname())
print('Python:', sys.executable)
print('code commit:', subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True).stdout.strip())
print('problem.py SHA256:', hashlib.sha256(Path('problem.py').read_bytes()).hexdigest())
print(f'{len(d)} original rays, noise sigma={P.SIGMA}; fixed disk mean={truth:.12f}')
print(f'disk centre={S.CENTRE}, radius={S.RADIUS}; fixed dimensionless lambda={S.LAMBDA}')
import pygeoinf0 as pg
print('pygeoinf0:',pg.__version__,pg.__file__)
print('settings:',pg.settings)


## Physical functionals and their Riesz representers

For a ray datum, $G_i(m)=\int_{\ell_i}m\,ds=\langle g_i,m\rangle_H$.
For the requested property, $T(m)=\int_{\rm disk}m\,dA/(\pi R^2)
=\langle t,m\rangle_H$.

We compute $K_{ij}=\langle g_i,g_j\rangle_H$ and $b_i=\langle t,g_i\rangle_H$.
The same constrained SOLA formula as in the pixel notebook then applies.
The dimensionless trade-off is fixed at 0.1; the accurately evaluated reference
$\|t\|_H^2$ is held fixed across quadrature levels.

Disk area integration is represented by Gauss quadrature over horizontal chords
with $y=c_y+R\sin\theta$. This is a numerical approximation of one fixed disk
functional. We check this approximation separately from ray-pair integration.

The smooth functions $g_i,t,r=\sum_i x_i g_i$ are **Riesz representers**. Their
pairing is the H2 inner product: their plotted values are not spatial weights
to integrate against the model using ordinary area measure.

In [ ]:
start=time.perf_counter()
rows,reference=S.sobolev_study()
S.save_json(OUT/'sobolev_results.json',dict(lambda_value=S.LAMBDA,rows=S.json_rows(rows),
          target_norm_squared=reference['tt'],target_chords=reference['target_nodes']))
np.savez_compressed(OUT/'sobolev_weights.npz',**{f"nodes{r['nodes']}":r['x'] for r in rows if 'x' in r})
print(f'Completed in {time.perf_counter()-start:.1f}s')
print('\n nodes   Gram rel.error  smallest eigenvalue   relative X change     estimate     std     relative mismatch')
for r in rows:
 if 'x' in r:
  print(f"{r['nodes']:5d} {r['gram_error']:13.3e} {r['min_eigenvalue']:19.3e} {r['x_error']:18.3e} "
        f"{r['estimate']:12.6f} {r['std']:9.3g} {r['relative_mismatch']:12.5g}")
 else: print(f"{r['nodes']:5d} {r['gram_error']:13.3e} {r['min_eigenvalue']:19.3e}   {r['status']}")

## Quadrature convergence

A Gram matrix must be positive semidefinite. A substantially negative eigenvalue
flags under-resolved pair integrals; those rows are not presented as valid SOLA
solutions. Agreement with the finest setting is a numerical check, not a theorem
that the reference is exact.

In [ ]:
good=[r for r in rows if 'x' in r]
fig,axs=plt.subplots(1,3,figsize=(13,3.5),constrained_layout=True)
axs[0].semilogy([r['nodes'] for r in rows],[max(r['gram_error'],1e-17) for r in rows],'o-')
axs[0].set_title('Gram change from 24-node reference')
axs[1].semilogy([r['nodes'] for r in good],[max(r['x_error'],1e-17) for r in good],'o-')
axs[1].set_title('relative weight change')
axs[2].plot([r['nodes'] for r in good],[r['estimate'] for r in good],'o-',label='estimate')
axs[2].plot([r['nodes'] for r in good],[r['delivered_truth'] for r in good],'s--',label='delivered truth')
axs[2].axhline(truth,color='k',label='target truth'); axs[2].legend(fontsize=8)
for ax in axs: ax.set_xlabel('Gauss nodes per integration piece'); ax.grid(alpha=.2)
plt.show()

## The same disk target at finer target quadrature

The primary convergence study used 32 target chords. Recompute only the target
pairings with 64 chords, keeping the ray-pair quadrature at 24. This isolates
target integration error from sensitivity-kernel integration error.

In [ ]:
fine_target=S.sobolev_gram(24,target_nodes=64)
xf=S.sola_weights(fine_target['K'],fine_target['b'],S.LAMBDA*reference['tt'],np.linalg.norm(q-p,axis=1))
target_check=dict(relative_X_change=float(np.linalg.norm(xf-reference['x'])/np.linalg.norm(reference['x'])),
                 relative_target_norm_change=float(abs(fine_target['tt']-reference['tt'])/reference['tt']),
                 target_truth_change=abs(S.target_truth(64)-S.target_truth(32)))
print(json.dumps(target_check,indent=2)); S.save_json(OUT/'target_quadrature_check.json',target_check)
assert target_check['relative_X_change'] < 2e-3, 'Target quadrature needs further refinement.'
assert target_check['target_truth_change'] < 1e-9

## Smooth Riesz functions and the underlying ray functional

Left and middle: target and delivered Riesz representers in H2, on the same
colour scale. Right: the original weighted rays. These are two representations
of functionals, not interchangeable spatial density pictures. In particular,
$\langle r,m\rangle_H=\sum_i x_i\int_{\ell_i}m\,ds$; it is generally not
$\int r m\,dA$. The model is not reconstructed in this experiment.

In [ ]:
start=time.perf_counter()
X,Y,rv,tv,rfun,tfun=S.sobolev_fields(reference,n=64)
fig,axs=plt.subplots(1,3,figsize=(14,4),constrained_layout=True)
lim=max(abs(rv).max(),abs(tv).max())
for ax,V,title in zip(axs[:2],[tv,rv],['target Riesz representer','delivered Riesz representer']):
 im=ax.imshow(V,origin='lower',extent=[0,1,0,1],cmap='RdBu_r',vmin=-lim,vmax=lim)
 ax.add_patch(plt.Circle(S.CENTRE,S.RADIUS,fill=False,color='k')); ax.set_title(title)
fig.colorbar(im,ax=axs[:2],shrink=.8)
lc=LineCollection(np.stack([p,q],axis=1),array=reference['x'],cmap='RdBu_r',linewidths=.7)
axs[2].add_collection(lc); axs[2].set_title('delivered functional: weighted original rays')
fig.colorbar(lc,ax=axs[2],shrink=.8,label='ray weight')
for ax in axs: ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_aspect('equal')
plt.show(); print(f'Field evaluation: {time.perf_counter()-start:.1f}s')
with S.quadrature_nodes(24):
 alone=float(np.asarray(rfun(np.array([.3]),np.array([.65]))).ravel()[0])
 two=np.asarray(rfun(np.array([.3,.4]),np.array([.65,.5]))).ravel()[0]
np.testing.assert_allclose(alone,two,rtol=2e-12,atol=1e-10)
print('Drawing-grid independence checked at a fixed physical point.')

## What the estimate establishes

The same noisy data are used throughout. The target bias below is known only
because this is a synthetic experiment. It is not included in the propagated
noise standard deviation.

In [ ]:
last=next(r for r in rows if r['nodes']==24)
for key in ['mass','estimate','delivered_truth','target_truth','target_bias','noise_draw','std','relative_mismatch']:
 print(f'{key:22s} {last[key]:.10g}')
np.testing.assert_allclose(last['estimate']-truth,last['target_bias']+last['noise_draw'],atol=2e-13)
if (OUT/'pixel_results.json').exists():
 pix=json.loads((OUT/'pixel_results.json').read_text())['rows']
 print('\n                       estimate    delivered truth    noise sigma    target bias')
 for label,z in [('pixels, finest',pix[-1]),('Sobolev, converged',last)]:
  print(f"{label:22s} {z['estimate']:10.6f} {z['delivered_truth']:16.6f} {z['std']:14.6g} {z['target_bias']:13.6f}")
 print('These use different mismatch geometries: this is not an accuracy ranking of one common estimator.')

## Interpretation and limits

When the weight vector stabilizes as quadrature is refined, the selected
transformation of the original MDR stabilizes. Its delivered property still
differs from the disk mean; convergence of numerical calculation does not remove
non-identifiability or turn propagated noise into a target-error bound.

The Sobolev metric provides a defined dual norm for the original ray functionals.
No norm bound or prior distribution is used. The choice of space and inner product
is explicit, and changing it changes which reachable question is judged closest.

Pixel-first L2 and continuum H2 therefore need not produce the same X. A growing
pixel density norm does not imply that every finite data combination is invalid:
its exact interpretation is always through its original weighted-ray MDR. It does
mean that such pixel kernels should not be presented as converging L2 sensitivity
representers without evidence and a compatible continuum formulation.

This notebook checks one target, one ray geometry and one fixed trade-off. It
does not prove that any target can be recovered, establish coverage after
data-dependent tuning, or show that pixel discretisations of an explicitly posed
Sobolev problem would fail. Those would approximate the Sobolev problem instead.